In [1]:
# Import packages
import numpy as np
import pandas as pd
import anndata
import scanpy as sc
import matplotlib.pyplot as plt
import re
import os
import sys
from scipy.sparse import csr_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
import matplotlib
import harmony

findfont: Font family ['Raleway'] not found. Falling back to DejaVu Sans.
findfont: Font family ['Lato'] not found. Falling back to DejaVu Sans.


In [2]:
# Functions 
'''CONSTRUCT AFFINITY MATRIX'''
def _convert_to_affinity(adj, scaling_factors, device, with_self_loops=False):
    """ Convert adjacency matrix to affinity matrix
    """
    N = adj.shape[0]
    rows, cols, dists = find(adj)
    if device == "gpu":
        import cupy as cp
        from cupyx.scipy.sparse import csr_matrix as csr_matrix_gpu
        dists = cp.array(dists) ** 2/(cp.array(scaling_factors.values[rows]) ** 2)
        rows, cols = cp.array(rows), cp.array(cols)
        # Self loops
        if with_self_loops:
            dists = cp.append(dists, cp.zeros(N))
            rows = cp.append(rows, range(N))
            cols = cp.append(cols, range(N))
        aff = csr_matrix_gpu((cp.exp(-dists), (rows, cols)), shape=(N, N)).get()
    elif device == "cpu":
        dists = dists ** 2/(scaling_factors.values[rows] ** 2)

        # Self loops
        if with_self_loops:
            dists = np.append(dists, np.zeros(N))
            rows = np.append(rows, range(N))
            cols = np.append(cols, range(N))
        aff = csr_matrix((np.exp(-dists), (rows, cols)), shape=[N, N])
    return aff

'''CONSTRUCT MUTUAL NEAREST NEIGHBORS GRAPH'''
def _construct_mnn(t1_cells, t2_cells, data_df, n_neighbors,device,n_jobs=-2):
    # FUnction to construct mutually nearest neighbors bewteen two points
    
    if device == "gpu":
        from cuml import NearestNeighbors
        nbrs = NearestNeighbors(n_neighbors=n_neighbors,
                                metric='cosine')
    elif device == "cpu":
        from sklearn.neighbors import NearestNeighbors
        nbrs = NearestNeighbors(n_neighbors=n_neighbors,
                                metric='cosine', n_jobs=n_jobs)
    
    print(f't+1 neighbors of t...')
    nbrs.fit(data_df.loc[t1_cells, :].values)
    t1_nbrs = nbrs.kneighbors_graph(
        data_df.loc[t2_cells, :].values, mode='distance')

    print(f't neighbors of t+1...')
    nbrs.fit(data_df.loc[t2_cells, :].values)
    t2_nbrs = nbrs.kneighbors_graph(
        data_df.loc[t1_cells, :].values, mode='distance')

    # Mututally nearest neighbors
    mnn = t2_nbrs.multiply(t1_nbrs.T)
    mnn = mnn.sqrt()
    return mnn

'''COMPUTE SCALING FACTORS'''
def _mnn_scaling_factors(mnn_ka_dists, scaling_factors,device):
    if device == "gpu":
        from cuml import LinearRegression
    else:
        from sklearn.linear_model import LinearRegression
    cells = mnn_ka_dists.index[~mnn_ka_dists.isnull()]

    # Linear model fit
    x = scaling_factors[cells]
    y = mnn_ka_dists[cells]
    lm = LinearRegression()
    lm.fit(x.values.reshape(-1, 1), y.values.reshape(-1, 1))

    # Predict
    x = scaling_factors[mnn_ka_dists.index]
    vals = np.ravel(lm.predict(x.values.reshape(-1, 1)))
    mnn_scaling_factors = pd.Series(vals, index=mnn_ka_dists.index)

    return mnn_scaling_factors

'''CONSTRUCT AFFINITY MATRIX'''
def _mnn_affinity(mnn, mnn_scaling_factors, offset1, offset2, device):
    # Function to convert mnn matrix to affinicty matrix

    # Construct adjacency matrix
    N = len(mnn_scaling_factors)
    x, y, z = find(mnn)
    x = x + offset1
    y = y + offset2
    adj = csr_matrix((z, (x, y)), shape=[N, N])

    # Affinity matrix
    return _convert_to_affinity(adj, mnn_scaling_factors, device, False)

'''
HARMONY AND PALANTIR
'''
def _mnn_ka_distances(mnn, n_neighbors):
    # Function to find distance ka^th neighbor in the mutual nearest neighbor matrix
    ka = int(n_neighbors / 3)
    ka_dists = np.repeat(None, mnn.shape[0])
    x, y, z = find(mnn)
    rows=pd.Series(x).value_counts()
    for r in rows.index[rows >= ka]:
        ka_dists[r] = np.sort(z[x==r])[ka - 1]
    return ka_dists

from harmony import core
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import find, csr_matrix

def harmony_aug_mat_with_pca(projections, timepoints, timepoint_connections, n_neighbors, n_neighbors2):
    # Time point cells and indices
    tp_cells = pd.Series()
    tp_offset = pd.Series()
    offset = 0
    for i in timepoints.unique():
        tp_offset[i] = offset
        tp_cells[i] = list(timepoints.index[timepoints == i])
        offset += len(tp_cells[i])

    # Nearest neighbor graph construction and affinity matrix
    print('Nearest neighbor computation...')
    nbrs = NearestNeighbors(n_neighbors=n_neighbors,
                            metric='cosine', n_jobs=-2).fit(projections.values)

    adj = nbrs.kneighbors_graph(projections.values, mode='distance')
    dists, _ = nbrs.kneighbors(projections.values)
    
    # Scaling factors for affinity matrix construction
    ka = int(n_neighbors / 3)
    scaling_factors = pd.Series(dists[:, ka], index=projections.index)
    
    # Affinity matrix
    nn_aff = _convert_to_affinity(adj, scaling_factors, 'cpu', True)
    n_jobs = -2
    
    # Mututally nearest neighbor affinity matrix
    # Initilze mnn affinity matrix
    N = projections.shape[0]
    full_mnn_aff = csr_matrix(([0], ([0], [0])), [N, N])
    for i in timepoint_connections.index:
        t1, t2 = timepoint_connections.loc[i, :].values
        print(f'Constucting affinities between {t1} and {t2}...')
        # MNN matrix  and distance to ka the distance
        t1_cells = tp_cells[t1]
        t2_cells = tp_cells[t2]
        mnn = _construct_mnn(t1_cells, t2_cells, projections,
                             n_neighbors2, 'cpu', n_jobs)
        
        # MNN Scaling factors
        # Distance to the adaptive neighbor
        ka_dists = pd.Series(0.0, index=t1_cells + t2_cells)
        ka_dists = ka_dists.astype(float)
        # T1 scaling factors
        ka_dists[t1_cells] = _mnn_ka_distances(mnn, n_neighbors2)
        # T2 scaling factors
        ka_dists[t2_cells] = _mnn_ka_distances(mnn.T, n_neighbors2)
        # Scaling factors
        mnn_scaling_factors = pd.Series(0.0, index=projections.index)
#         mnn_scaling_factors[t1_cells] = core._mnn_scaling_factors(
#             ka_dists[t1_cells], scaling_factors,'cpu')
#         mnn_scaling_factors[t2_cells] = core._mnn_scaling_factors(
#             ka_dists[t2_cells], scaling_factors,'cpu')
        mnn_scaling_factors[t1_cells] = _mnn_scaling_factors(
            ka_dists[t1_cells], scaling_factors,'cpu')
        mnn_scaling_factors[t2_cells] = _mnn_scaling_factors(
            ka_dists[t2_cells], scaling_factors,'cpu')
        
        # MNN affinity matrix
#         full_mnn_aff = full_mnn_aff + \
#             core._mnn_affinity(mnn, mnn_scaling_factors,
#                           tp_offset[t1], tp_offset[t2], 'cpu')
        full_mnn_aff = full_mnn_aff + \
            _mnn_affinity(mnn, mnn_scaling_factors,
                          tp_offset[t1], tp_offset[t2], 'cpu')
    # Symmetrize the affinity matrix and return
    aug_aff2 = nn_aff + nn_aff.T + full_mnn_aff + full_mnn_aff.T
    aff2 = nn_aff + nn_aff.T
    return aug_aff2, aff2

'''COMPUTE RANDOM WALK PROBABILITIES'''
def random_walk_probabilities(A, labels):
    D = np.diag(np.sum(A, axis=1))
    L = D - A  # graph laplacian
    seeds = np.array([e != 0 for e in labels], dtype=bool)
    Lu = L[np.invert(seeds),:][:, np.invert(seeds)]  # unlabeled rows, unlabeled cols
    BT = L[np.invert(seeds),:][:, seeds]  # unlabeled rows, labeled cols
    classes = np.unique(labels[labels > 0])
    M = np.zeros((seeds.sum(), len(classes)))
    for k in classes:
        M[labels[seeds] == k, k-1] = 1
    P = np.linalg.lstsq(Lu, np.dot(-BT, M), rcond = None)[0]
    return P


In [3]:
# Set plotting settings
sns.set_style('white')
matplotlib.rcParams['figure.figsize'] = [4, 4]
matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['image.cmap'] = 'Spectral_r'
matplotlib.rcParams['savefig.dpi'] = 150
matplotlib.style.use("ggplot")
warnings.filterwarnings(action="ignore", module="matplotlib", message="findfont")


In [9]:
# Set the directory containing the .h5ad files
input_dir = "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_noreplicates/stratify/"

# Extract sample name from the command line argument
sample = 'Metastatic_BASE'

# Construct the file path for the sample
file_path = os.path.join(input_dir, f"{sample}.h5ad")

In [10]:
# Load data
adata_org2 = sc.read_h5ad(file_path)

# Load patient dataset (which contains the patient labels)
adata_patient = sc.read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Patients.HTAN/adatas/KG146_Patient_Organoid.h5ad')

In [11]:
adata_patient

AnnData object with n_obs × n_vars = 12016 × 20157
    obs: 'n_counts', 'n_genes', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'original_total_counts', 'log10_original_total_counts', 'original_mito_counts', 'log10_original_mito_counts', 'Sample ID', 'emptyDrops_Total', 'emptyDrops_LogProb', 'emptyDrops_PValue', 'emptyDrops_FDR', 'latent_cell_probability', 'PhenoGraph_clusters', 'Smilie Cell Type', 'log10_n_counts', 'frac_counts_gt1_reads', 'n_reads', 'n_reads_per_count', 'n_counts_vs_ED_LogProb', 'n_counts_vs_n_genes_by_counts', 'n_flagged_metrics', 'DC 1', 'DC 2', 'DC 3', 'DC 4', 'DC 5', 'DC 6', 'DC 7', 'Palantir Diff. Potential', 'Palantir Pseudotime', 'Combined Fetal Signature', 'First Tri